# Tier 5 Assignments — Set 1: Modern AI for Science

**Modules covered**: LLM Fine-tuning (LoRA), Retrieval-Augmented Generation (RAG), Diffusion & Generative Models

---

## Grading Rubric

| Problem | Topic | Stars | Points |
|---------|-------|-------|--------|
| 1 | LoRA Rank Decomposition | ★ | 10 |
| 2 | LoRA Parameter Efficiency | ★ | 10 |
| 3 | Chat Template Formatting | ★★ | 10 |
| 4 | TF-IDF Document Retrieval | ★★ | 15 |
| 5 | DDPM Forward Noise Process | ★ | 15 |
| 6 | Linear and Cosine Noise Schedules | ★★ | 15 |
| 7 | Cosine Similarity for Multi-Vector Retrieval | ★★★ | 10 |
| 8 | DDIM Deterministic Sampling Step | ★★★ | 15 |
| **Total** | | | **100** |

---

**All problems use pure NumPy and the Python standard library — no GPU or large model inference required.**

In [ ]:
import numpy as np
import math
from math import log, sqrt, cos, pi
import random

## Problem 1: LoRA Rank Decomposition (1 star, 10 pts)

Low-Rank Adaptation (LoRA) decomposes a weight update ΔW into ΔW = B @ A, where B is (d_out, r) and A is (r, d_in) with rank r << d. This dramatically reduces the number of trainable parameters during fine-tuning.

Given a full weight matrix W (2D numpy array), approximate it with a rank-r SVD:

W ≈ U[:, :r] @ diag(S[:r]) @ Vt[:r, :]

Compute the LoRA factor matrices:
- B = U[:, :r] * sqrt(S[:r])   — shape (d_out, r)
- A = diag(sqrt(S[:r])) @ Vt[:r, :]   — shape (r, d_in)

Return a dict `{"B": array, "A": array, "reconstruction_error": float}` where `reconstruction_error` is the Frobenius norm of (W - B @ A).

**Grading**: correct shapes for B and A (3 pts), B @ A approximates W (4 pts), reconstruction error computed correctly (3 pts)

In [ ]:
def lora_decompose(W: np.ndarray, r: int) -> dict:
    """
    Approximate weight matrix W with a rank-r LoRA decomposition.

    Args:
        W: Weight matrix of shape (d_out, d_in).
        r: LoRA rank (must be <= min(d_out, d_in)).

    Returns:
        Dict with keys 'B' (d_out, r), 'A' (r, d_in), 'reconstruction_error' (float).
    """
    # YOUR CODE HERE
    pass


rng = np.random.default_rng(42)
W_test = rng.standard_normal((8, 6))
result = lora_decompose(W_test, r=2)
print(f"B shape: {result['B'].shape}")
print(f"A shape: {result['A'].shape}")
print(f"Reconstruction error (r=2): {result['reconstruction_error']:.4f}")

# Full rank should give near-zero error
result_full = lora_decompose(W_test, r=6)
print(f"Reconstruction error (r=6, full): {result_full['reconstruction_error']:.6f}")

## Problem 2: LoRA Parameter Efficiency (1 star, 10 pts)

One of LoRA's key benefits is drastically reducing the number of trainable parameters. Given a transformer model's dimensions, compute how many parameters are in the original attention weight matrices versus the LoRA adapters.

Assume:
- 4 attention weight matrices per layer: Q, K, V, O — each of shape (d_model × d_model)
- LoRA is applied only to Q and V (common practice), each needing matrices A and B
- Each LoRA pair adds r * d_model (for A) + d_model * r (for B) = 2 * r * d_model parameters

Compute:
- `original_params`: total params in all 4 attention matrices across all layers
- `lora_params`: total LoRA parameters (Q and V only, both A and B) across all layers
- `compression_ratio`: lora_params / original_params

Return `{"original_params": int, "lora_params": int, "compression_ratio": float}`.

**Grading**: original_params correct (3 pts), lora_params correct (4 pts), compression_ratio correct (3 pts)

In [ ]:
def lora_efficiency(d_model: int, num_layers: int, lora_rank: int) -> dict:
    """
    Compute LoRA parameter efficiency compared to full fine-tuning of attention weights.

    Args:
        d_model: Hidden size (dimension of each attention matrix side).
        num_layers: Number of transformer layers.
        lora_rank: LoRA rank r.

    Returns:
        Dict with 'original_params', 'lora_params', 'compression_ratio'.
    """
    # YOUR CODE HERE
    pass


# LLaMA-7B-style model
result = lora_efficiency(d_model=4096, num_layers=32, lora_rank=8)
print(f"Original attention params : {result['original_params']:,}")
print(f"LoRA params               : {result['lora_params']:,}")
print(f"Compression ratio         : {result['compression_ratio']:.4f}")
print(f"Parameter reduction       : {1/result['compression_ratio']:.1f}x")

## Problem 3: Chat Template Formatting (2 stars, 10 pts)

Large language models are fine-tuned to follow specific chat templates that structure the conversation into system, user, and assistant turns. Getting the template exactly right is critical — even small formatting differences can degrade model quality.

Implement the **LLaMA-3 chat template**. Given a list of messages, format them as follows:

- If the first message has role `"system"`:
  `<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{content}<|eot_id|>`
- User turn:
  `<|start_header_id|>user<|end_header_id|>\n\n{content}<|eot_id|>`
- Assistant turn:
  `<|start_header_id|>assistant<|end_header_id|>\n\n{content}<|eot_id|>`

If there is no system message, start with `<|begin_of_text|>` before the first user turn.

At the very end, always append `<|start_header_id|>assistant<|end_header_id|>\n\n` (no `<|eot_id|>`) to prompt the model to generate its next response.

Return the fully formatted string.

**Grading**: correct special tokens (4 pts), system/user/assistant handling (3 pts), trailing generation prompt (3 pts)

In [ ]:
def format_llama3_chat(messages: list[dict]) -> str:
    """
    Format a list of messages using the LLaMA-3 chat template.

    Args:
        messages: List of dicts with 'role' ('system', 'user', 'assistant')
                  and 'content' (str).

    Returns:
        Formatted prompt string ready for model input.
    """
    # YOUR CODE HERE
    pass


messages = [
    {"role": "system", "content": "You are a helpful bioinformatics assistant."},
    {"role": "user", "content": "What is a FASTA file?"},
    {"role": "assistant", "content": "A FASTA file stores biological sequences with a header line starting with >."},
    {"role": "user", "content": "How do I parse one in Python?"},
]

formatted = format_llama3_chat(messages)
print(repr(formatted))
print()
print(formatted)

## Problem 4: TF-IDF Document Retrieval (2 stars, 15 pts)

Retrieval-Augmented Generation (RAG) systems first retrieve relevant documents before generating an answer. The classic baseline is TF-IDF (Term Frequency–Inverse Document Frequency).

Implement TF-IDF retrieval. Given a corpus (list of documents as strings) and a query string, compute relevance scores and return the top-k document indices.

Formulas (case-insensitive, split on whitespace):
- TF(term, doc) = count of term in doc / total number of terms in doc
- IDF(term) = log((1 + N) / (1 + df(term))) + 1, where N = len(corpus), df(term) = number of docs containing term
- Score(doc) = sum of TF(term, doc) * IDF(term) for each query term present in doc

Return the top-k document indices sorted by score descending (as a list of ints).

**Grading**: TF computation (4 pts), IDF computation (4 pts), final score and ranking (4 pts), edge cases (3 pts)

In [ ]:
def tfidf_retrieve(corpus: list[str], query: str, k: int = 3) -> list[int]:
    """
    Retrieve the top-k most relevant document indices using TF-IDF scoring.

    Args:
        corpus: List of document strings.
        query: Query string.
        k: Number of top documents to return.

    Returns:
        List of up to k document indices sorted by descending TF-IDF score.
    """
    # YOUR CODE HERE
    pass


corpus = [
    "CRISPR is a gene editing tool derived from bacterial immune systems.",
    "RNA sequencing measures gene expression levels across the transcriptome.",
    "CRISPR Cas9 enables precise gene editing by cutting DNA at target sites.",
    "Single cell RNA sequencing reveals cell type heterogeneity in tissues.",
    "Protein folding determines the three-dimensional structure of proteins.",
    "AlphaFold predicts protein structure from amino acid sequence using deep learning.",
]

query = "CRISPR gene editing"
top_k = tfidf_retrieve(corpus, query, k=3)
print(f"Query: {query!r}")
print(f"Top-{len(top_k)} results:")
for idx in top_k:
    print(f"  [{idx}] {corpus[idx]}")

## Problem 5: DDPM Forward Noise Process (1 star, 15 pts)

Diffusion models learn to reverse a gradual noising process. The forward process adds Gaussian noise to a clean sample over T timesteps. The key insight is that the noisy sample at any timestep t can be computed in one shot using the **closed-form formula**:

x_t = sqrt(ᾱ_t) * x_0 + sqrt(1 - ᾱ_t) * ε

where:
- ᾱ_t = product of (1 - β_i) for i = 1, 2, ..., t
- ε ~ N(0, I) is standard Gaussian noise
- β values are the noise schedule (given as a list, 0-indexed: β[0] is β_1)

Use `np.random.seed(seed)` before sampling ε for reproducibility.

Return `{"x_t": array, "alpha_bar_t": float, "noise": array}`.

**Grading**: correct ᾱ_t computation (5 pts), correct closed-form formula for x_t (7 pts), reproducible noise (3 pts)

In [ ]:
def ddpm_forward(x0: np.ndarray, betas: list[float], t: int, seed: int = 0) -> dict:
    """
    Apply the DDPM forward diffusion process to get the noisy sample at timestep t.

    Args:
        x0: Clean sample, 1D numpy array.
        betas: Noise schedule, list of T beta values (0-indexed, beta[0] = beta_1).
        t: Timestep (1-indexed, so t=1 uses betas[0]).
        seed: Random seed for reproducibility.

    Returns:
        Dict with 'x_t', 'alpha_bar_t', 'noise'.
    """
    # YOUR CODE HERE
    pass


x0 = np.array([1.0, 0.5, -0.3, 0.8])
T = 10
betas = [0.001 + 0.001 * i for i in range(T)]  # simple linear schedule

for t in [1, 5, 10]:
    res = ddpm_forward(x0, betas, t, seed=42)
    print(f"t={t:2d}: alpha_bar={res['alpha_bar_t']:.4f}  x_t={np.round(res['x_t'], 4)}")

## Problem 6: Linear and Cosine Noise Schedules (2 stars, 15 pts)

The noise schedule (how quickly noise is added) has a large impact on diffusion model quality. Two widely used schedules are:

**Linear** (DDPM original):
- β_t = β_start + (β_end - β_start) * t / (T - 1), for t = 0, 1, ..., T-1

**Cosine** (improved DDPM):
- f(t) = cos²(((t / T + s) / (1 + s)) * π/2), with s = 0.008
- ᾱ_t = f(t) / f(0)
- β_t = min(1 - ᾱ_t / ᾱ_{t-1}, 0.999), for t = 1, ..., T
- Set β_0 = 1 - ᾱ_0 (the first beta, with ᾱ_{-1} = 1)

For both schedules, also compute the cumulative product ᾱ_t = product of (1 - β_i) for i = 0..t.

Return a dict with keys `"linear_betas"`, `"cosine_betas"`, `"linear_alpha_bars"`, `"cosine_alpha_bars"` — all numpy arrays of length T.

**Grading**: linear betas correct (3 pts), cosine betas correct (5 pts), alpha_bars from cumprod (4 pts), correct shapes (3 pts)

In [ ]:
def noise_schedules(T: int, beta_start: float = 1e-4, beta_end: float = 0.02) -> dict:
    """
    Compute linear and cosine noise schedules.

    Args:
        T: Number of diffusion timesteps.
        beta_start: Starting beta for linear schedule.
        beta_end: Ending beta for linear schedule.

    Returns:
        Dict with 'linear_betas', 'cosine_betas', 'linear_alpha_bars', 'cosine_alpha_bars'.
    """
    # YOUR CODE HERE
    pass


schedules = noise_schedules(T=1000)
print(f"Linear betas  — first: {schedules['linear_betas'][0]:.6f}  last: {schedules['linear_betas'][-1]:.6f}")
print(f"Cosine betas  — first: {schedules['cosine_betas'][0]:.6f}  last: {schedules['cosine_betas'][-1]:.6f}")
print(f"Linear alpha_bars — first: {schedules['linear_alpha_bars'][0]:.6f}  last: {schedules['linear_alpha_bars'][-1]:.6f}")
print(f"Cosine alpha_bars — first: {schedules['cosine_alpha_bars'][0]:.6f}  last: {schedules['cosine_alpha_bars'][-1]:.6f}")

## Problem 7: Cosine Similarity for Multi-Vector Retrieval (3 stars, 10 pts)

ColPali and similar Vision-RAG systems represent each document as a **set of patch embeddings** (one per image patch or token) rather than a single vector. Retrieval then uses the **MaxSim** score: for each query token, find its best-matching document token, then average across all query tokens.

Given:
- Q: query token matrix, shape (q_tokens, dim)
- D: document token matrix, shape (d_tokens, dim)

Compute:

MaxSim(Q, D) = mean over i of ( max over j of ( cosine_similarity(Q[i], D[j]) ) )

where cosine_similarity(u, v) = dot(u, v) / (||u|| * ||v||).

Handle the case where a vector has zero norm (return 0 similarity for that pair).

Return the MaxSim score as a float.

**Grading**: cosine similarity computation (4 pts), max over document tokens (3 pts), mean over query tokens (3 pts)

In [ ]:
def maxsim(Q: np.ndarray, D: np.ndarray) -> float:
    """
    Compute the MaxSim score between a query and a document using multi-vector representations.

    Args:
        Q: Query token embeddings, shape (q_tokens, dim).
        D: Document token embeddings, shape (d_tokens, dim).

    Returns:
        MaxSim score as a float.
    """
    # YOUR CODE HERE
    pass


rng = np.random.default_rng(7)
Q = rng.standard_normal((4, 8))   # 4 query tokens, dim=8
D = rng.standard_normal((10, 8))  # 10 document tokens, dim=8

score = maxsim(Q, D)
print(f"MaxSim score: {score:.4f}")

# Identical query and document (each row is the same) should give score ~ 1.0
identical = rng.standard_normal((1, 8))
same_score = maxsim(np.tile(identical, (3, 1)), np.tile(identical, (3, 1)))
print(f"MaxSim (identical Q=D): {same_score:.4f}  (expected ~1.0)")

## Problem 8: DDIM Deterministic Sampling Step (3 stars, 15 pts)

DDIM (Denoising Diffusion Implicit Models) provides a **deterministic** alternative to DDPM sampling, allowing for faster generation with fewer steps by skipping timesteps.

Implement one DDIM sampling step. Given the current noisy sample x_t and the model's predicted noise ε_θ(x_t, t), compute x_{t_prev} using:

1. **Predict x_0**:  
   predicted_x0 = (x_t - sqrt(1 - ᾱ_t) * ε_θ) / sqrt(ᾱ_t)

2. **Direction toward x_t**:  
   direction_to_xt = sqrt(1 - ᾱ_{t_prev}) * ε_θ

3. **Update**:  
   x_{t_prev} = sqrt(ᾱ_{t_prev}) * predicted_x0 + direction_to_xt

Note: ᾱ values are indexed by timestep — use `alpha_bars[t]` and `alpha_bars[t_prev]`.

Return `{"x_t_prev": array, "predicted_x0": array}`.

**Grading**: predicted_x0 formula (4 pts), direction_to_xt formula (3 pts), x_{t_prev} update (5 pts), correct indexing (3 pts)

In [ ]:
def ddim_step(
    x_t: np.ndarray,
    predicted_noise: np.ndarray,
    t: int,
    t_prev: int,
    alpha_bars: list[float],
) -> dict:
    """
    Perform one deterministic DDIM denoising step.

    Args:
        x_t: Current noisy sample at timestep t.
        predicted_noise: Noise predicted by the model, same shape as x_t.
        t: Current timestep index (used to index alpha_bars).
        t_prev: Previous (less noisy) timestep index (t_prev < t).
        alpha_bars: List of cumulative alpha_bar values for all timesteps.

    Returns:
        Dict with 'x_t_prev' and 'predicted_x0', both numpy arrays.
    """
    # YOUR CODE HERE
    pass


# Build a simple cosine schedule for testing
T = 100
s = 0.008
alpha_bars_test = [
    (cos(((t / T + s) / (1 + s)) * pi / 2) ** 2) / (cos((s / (1 + s)) * pi / 2) ** 2)
    for t in range(T)
]

rng = np.random.default_rng(3)
x_t = rng.standard_normal(4)
eps = rng.standard_normal(4)

result = ddim_step(x_t, eps, t=80, t_prev=60, alpha_bars=alpha_bars_test)
print(f"x_t         : {np.round(x_t, 4)}")
print(f"predicted_x0: {np.round(result['predicted_x0'], 4)}")
print(f"x_t_prev    : {np.round(result['x_t_prev'], 4)}")

---

## Summary

| Problem | Topic | Stars | Points |
|---------|-------|-------|--------|
| 1 | LoRA Rank Decomposition | ★ | 10 |
| 2 | LoRA Parameter Efficiency | ★ | 10 |
| 3 | Chat Template Formatting | ★★ | 10 |
| 4 | TF-IDF Document Retrieval | ★★ | 15 |
| 5 | DDPM Forward Noise Process | ★ | 15 |
| 6 | Linear and Cosine Noise Schedules | ★★ | 15 |
| 7 | Cosine Similarity for Multi-Vector Retrieval | ★★★ | 10 |
| 8 | DDIM Deterministic Sampling Step | ★★★ | 15 |
| **Total** | | | **100** |